# Gradio Dashboard


## 1 · Imports, saved results and dashboard design

In [ ]:
"""Gradio dashboard for the frozen, committed YOLO11s hard-hat model.

No training, Roboflow access or dataset downloads occur at app startup.
Run via: uv run python main.py app
"""
from __future__ import annotations

import json
import os
import inspect
from pathlib import Path

import gradio as gr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from ultralytics import YOLO

"""Single source of truth for image-level triage and modeled economic costs.

Truth rows: 0 = no annotated exposed head; 1 = >=1 annotated exposed head.
Action columns: 0 = PASS; 1 = HUMAN REVIEW; 2 = FLAG.
Review is assumed error-free; reviewed positives still incur the €2 intervention.
"""


PASS, REVIEW, FLAG = 0, 1, 2


def policy_actions(df: pd.DataFrame, low: float, high: float, triage: bool = True) -> np.ndarray:
    if not (0 <= float(low) <= float(high) <= 1):
        raise ValueError("Expected 0 <= lower threshold <= upper threshold <= 1")
    scores = df["unsafe_score"].to_numpy(dtype=float)
    if not triage:
        return np.where(scores >= low, FLAG, PASS).astype(int)
    actions = np.where(scores >= high, FLAG, np.where(scores < low, PASS, REVIEW))
    # If even head AND helmet lack strong evidence, a PASS is unsafe to assert.
    # These image-level scores are not calibrated probabilities.
    return np.where(df["no_evidence"].to_numpy(dtype=bool), REVIEW, actions).astype(int)


def evaluate_actions(
    df: pd.DataFrame,
    actions: np.ndarray,
    prevalence: float,
    costs: dict[str, float],
) -> dict:
    if not 0 <= prevalence <= 1:
        raise ValueError("Prevalence must be in [0, 1]")
    truth = df["unsafe_gt"].to_numpy(dtype=int)
    actions = np.asarray(actions, dtype=int)
    if len(truth) != len(actions):
        raise ValueError("Action length does not match case count")
    if not np.isin(truth, (0, 1)).all() or not np.isin(actions, (0, 1, 2)).all():
        raise ValueError("Truth/actions contain unexpected labels")
    matrix = np.array([
        [int(((truth == i) & (actions == j)).sum()) for j in (PASS, REVIEW, FLAG)]
        for i in (0, 1)
    ], dtype=int)
    if (matrix.sum(axis=1) == 0).any():
        raise ValueError("Both truth classes must be represented")
    rates = matrix / matrix.sum(axis=1, keepdims=True)
    # Reviewed *positive* images still need the real-world intervention.
    # Omitting costs['tp'] here underprices a human-review policy.
    negative_costs = [costs['tn'], costs['review'], costs['fp']]
    positive_costs = [costs['fn'], costs['review'] + costs['tp'], costs['tp']]
    expected_cost = 1000 * (
        (1 - prevalence) * float(np.dot(rates[0], negative_costs))
        + prevalence * float(np.dot(rates[1], positive_costs))
    )
    review_share = float((1 - prevalence) * rates[0, REVIEW] + prevalence * rates[1, REVIEW])
    return {
        'cost_per_1000': float(expected_cost),
        'review_share': review_share,
        'matrix': matrix.tolist(),
        'rates_by_truth': rates.tolist(),
    }


def alternative_costs(prevalence: float, costs: dict[str, float]) -> dict[str, float]:
    return {
        'Do nothing (always pass)': 1000 * (prevalence * costs['fn'] + (1 - prevalence) * costs['tn']),
        'Person reviews all': 1000 * (costs['review'] + prevalence * costs['tp']),
    }

# Compatible with both VS Code/Jupyter cells and `main.py app` (nbconvert).
ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd().resolve()
ARTIFACTS = ROOT / 'artifacts'
M = json.loads((ARTIFACTS / 'metrics.json').read_text(encoding='utf-8'))
VAL = pd.read_csv(ARTIFACTS / 'val_cases.csv')
TEST = pd.read_csv(ARTIFACTS / 'test_cases.csv')
HEALTH = pd.read_csv(ARTIFACTS / 'dataset_health.csv')
WEIGHTS = ROOT / 'weights' / 'best.pt'
MODEL = YOLO(str(WEIGHTS))
COST = M['cost_euros']
HEAD, HELMET = M['class_ids']['head'], M['class_ids']['helmet']
NAMES = M['class_names']
SINGLE = M['single_threshold']

# The model, CSV data, displayed economics and threshold optimizer must agree.
assert M.get('decision_policy_version') == 2, (
    'Decision artifacts are outdated. Copy the corrected metrics.json and CSV files from the revised ZIP.'
)
recheck = evaluate_actions(
    TEST, policy_actions(TEST, M['low_threshold'], M['high_threshold']),
    M['base_rate_assumed'], COST,
)
assert abs(recheck['cost_per_1000'] - M['test_hybrid']['cost_per_1000']) < 1e-7, (
    'Saved artifacts disagree with dashboard economics.'
)

CSS = """
.gradio-container {max-width: 1440px !important; margin:auto !important; padding: 12px 26px 40px !important;}
body {background: #f5f7fb !important;}
#hero {background: linear-gradient(115deg, #102338 0%, #183d50 62%, #0e665e 100%);
       padding: 32px 36px; border-radius: 21px; color: white; box-shadow: 0 16px 30px #14293b16;}
#hero .eyebrow {font-size: 12px; letter-spacing: .16em; text-transform: uppercase; font-weight: 800; color:#91ead5;}
#hero h1 {font-size: 34px; letter-spacing: -.03em; color:white !important; margin: 7px 0 4px; font-weight:800;}
#hero p {font-size: 15px; color:#d4e4eb; margin: 3px 0;}
#hero .tag {display:inline-block; color:#dff9f3; font-size:11px; border:1px solid #5f9f9c;
           border-radius:99px; padding:4px 10px; margin-top:13px; margin-right:6px;}
#kpis {display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:13px;margin:7px 0 12px;}
.metric {background:white;padding:21px;border:1px solid #e3eaf1;border-radius:15px;box-shadow:0 5px 14px #132a3a08;}
.metric .m-label {color:#63788b;font-size:12px;font-weight:750;text-transform:uppercase;letter-spacing:.065em;}
.metric .m-value {color:#132e43;font-size:27px;font-weight:830;letter-spacing:-.035em;margin:9px 0 2px;}
.metric .m-sub {font-size:12px;color:#6b7e8d;}
.metric .ok {color:#006d57;}
.metric .alert {color:#ba4b17;}
.section-title {font-size:19px;font-weight:800;color:#18354b;margin:8px 0 0;}
#footnote {color:#6d7c88;font-size:12px;line-height:1.5;}
@media(max-width:950px){#kpis{grid-template-columns:repeat(2,minmax(0,1fr));}
 #hero h1{font-size:25px;} .gradio-container{padding:8px 12px 35px !important;}}
@media(max-width:540px){#kpis{grid-template-columns:1fr;} #hero{padding:22px;}}
"""


## 2 · Decision policy and economic visualizations

In [ ]:
def measure(df: pd.DataFrame, actions: np.ndarray, prevalence: float) -> dict:
    return evaluate_actions(df, actions, prevalence, COST)


def draw_matrix(cm: np.ndarray):
    # Review becomes the correct answer only under the explicit perfect-review assumption.
    corrected = np.array([
        [cm[0, 0] + cm[0, 1], cm[0, 2]],
        [cm[1, 0], cm[1, 1] + cm[1, 2]],
    ], dtype=int)
    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.2), constrained_layout=True)
    for ax, matrix, columns, title in [
        (axes[0], cm, ['Pass', 'Review', 'Flag'], 'Decision before human review'),
        (axes[1], corrected, ['Cleared', 'Flagged'], 'After review · idealized'),
    ]:
        ax.imshow(matrix, cmap='Blues', vmin=0, aspect='auto')
        ax.set_xticks(range(len(columns)), columns)
        ax.set_yticks([0, 1], ['No exposed head', 'Exposed head'])
        ax.set_xlabel('Policy decision', color='#43586b')
        ax.set_title(title, fontsize=12, weight='bold', color='#18354b')
        for (r, c), v in np.ndenumerate(matrix):
            ax.text(c, r, f'{v:,}', ha='center', va='center', fontsize=12, weight='semibold', color='#152a3c')
    return fig


def draw_sweep(prevalence: float, lower: float, upper: float):
    grid = np.round(np.linspace(.01, 1, 100), 3)
    costs = np.array([
        measure(VAL, policy_actions(VAL, t, t, False), prevalence)['cost_per_1000']
        for t in grid
    ])
    fig, ax = plt.subplots(figsize=(10.5, 4.15), constrained_layout=True)
    ax.plot(grid, costs, lw=2.5, color='#006d61', label='Model alone · validation')
    ax.axvspan(lower, upper, color='#f2b770', alpha=.28, label='Current review interval')
    ax.axvline(SINGLE, color='#142d44', ls='--', lw=1.7, label=f'Model-alone chosen cutoff {SINGLE:.2f}')
    ax.axvline(lower, color='#9d6121', ls=':', lw=1.5, label=f'Hybrid lower {lower:.3f}')
    ax.axvline(upper, color='#ac6535', ls=':', lw=1.5, label=f'Hybrid upper {upper:.3f}')
    best = int(np.argmin(costs))
    ax.scatter([grid[best]], [costs[best]], s=65, zorder=3, color='#e08035',
               label=f'Best at current prevalence {grid[best]:.2f}')
    ax.set(xlabel='Exposed-head confidence cutoff', ylabel='Expected €/1,000 images',
           title='Cost versus threshold · validation only')
    ax.set_title('Cost versus threshold · validation only', fontsize=12, weight='bold', color='#18354b')
    ax.grid(alpha=.16)
    ax.legend(loc='best', fontsize=8, frameon=False)
    return fig


def draw_action_mix(rates: np.ndarray, prevalence: float):
    props = ((1-prevalence) * rates[0] + prevalence * rates[1])
    fig, ax = plt.subplots(figsize=(5, 2.5), constrained_layout=True)
    left = 0
    for p, label, color in zip(props, ['Pass', 'Review', 'Flag'], ['#5d9b8e', '#e1ad57', '#cc6455']):
        ax.barh(['Expected actions'], [p], left=left, color=color, height=.48, label=f'{label}: {p:.1%}')
        if p >= .07:
            ax.text(left + p/2, 0, f'{p:.1%}', va='center', ha='center', color='white', weight='bold', fontsize=11)
        left += p
    ax.set_xlim(0, 1)
    ax.set_xticks([0, .25, .5, .75, 1], ['0%', '25%', '50%', '75%', '100%'])
    ax.set_title('Expected policy mix · adjusted for prevalence', fontsize=12, color='#18354b')
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.35), ncol=3, fontsize=9, frameon=False)
    ax.spines[['left','right','top']].set_visible(False)
    return fig


def kpi_html(expected: float, review: float, minutes: float, prevalence: float) -> str:
    reviews_day = review * M['cases_per_day']
    remaining = M['review_minutes_available'] - minutes
    status = f'{remaining:,.0f} min spare' if remaining >= 0 else f'{-remaining:,.0f} min over budget'
    status_class = 'ok' if remaining >= 0 else 'alert'
    return f'''<div id="kpis">
      <div class="metric"><div class="m-label">Expected cost</div><div class="m-value">€{expected:,.0f}</div>
        <div class="m-sub">per 1,000 images · {prevalence:.0%} assumed prevalence</div></div>
      <div class="metric"><div class="m-label">Human review share</div><div class="m-value">{review:.1%}</div>
        <div class="m-sub">{reviews_day:.1f} cases per 600-image day</div></div>
      <div class="metric"><div class="m-label">Reviewer workload</div><div class="m-value">{minutes:.0f} min</div>
        <div class="m-sub">{minutes/M['fte_minutes_per_day']:.2f} FTE equivalent per day</div></div>
      <div class="metric"><div class="m-label">120-minute daily capacity</div>
        <div class="m-value {status_class}">{status}</div><div class="m-sub">At 45 seconds per review</div></div>
    </div>'''


def refresh(lower: float, upper: float, prior_percent: float):
    lo, hi = sorted((float(lower), float(upper)))
    prior = float(prior_percent) / 100
    result = measure(TEST, policy_actions(TEST, lo, hi), prior)
    model = measure(TEST, policy_actions(TEST, SINGLE, SINGLE, False), prior)
    cm = np.asarray(result['matrix'])
    rates = np.asarray(result['rates_by_truth'])
    review = result['review_share']
    minutes = review * M['cases_per_day'] * M['review_seconds'] / 60
    base = alternative_costs(prior, COST)
    alternative_rows = [
        ['Do nothing · always pass', base['Do nothing (always pass)'], 0.0, 'Yes'],
        ['Human reviews every image', base['Person reviews all'], 1.0, 'No · 450 min/day'],
        [f'Model alone · cutoff {SINGLE:.2f}', model['cost_per_1000'], 0.0, 'Yes'],
        ['Hybrid · current thresholds', result['cost_per_1000'], review,
         'Yes' if minutes <= M['review_minutes_available'] else 'No · over capacity'],
    ]
    alternatives = pd.DataFrame(alternative_rows, columns=['Strategy', 'Expected €/1,000', 'Review share', 'Fits allocated 120 min?'])
    alternatives['Expected €/1,000'] = alternatives['Expected €/1,000'].round(2)
    alternatives['Review share'] = alternatives['Review share'].map(lambda x: f'{x:.1%}')
    order_note = 'Threshold order automatically sorted. ' if lower > upper else ''
    notes = (
        f'**Current policy:** PASS below {lo:.3f}, REVIEW from {lo:.3f} to {hi:.3f}, '
        f'FLAG at or above {hi:.3f}. Low head **and** helmet evidence always triggers review.  '
        f'{order_note}\n\n'
        f'**Raw held-out test sample:** {len(TEST)} images; {int((TEST.unsafe_gt==1).sum())} annotated-positive. '
        f'Confusion matrix cells are raw test counts; **expected cost and workload are prevalence-adjusted**. '
        'Human review is assumed correct, and reviewed positive images incur both review and intervention costs.'
    )
    return (
        kpi_html(result['cost_per_1000'], review, minutes, prior),
        notes,
        draw_matrix(cm),
        draw_sweep(prior, lo, hi),
        draw_action_mix(rates, prior),
        alternatives,
    )


## 3 · Photograph inspection and evaluation summaries

In [ ]:
def inference_once(image: Image.Image | None):
    if image is None:
        return None
    rgb = image.convert('RGB')
    result = MODEL.predict(
        source=rgb, imgsz=M['imgsz'], conf=M['inference_conf'],
        iou=M['nms_iou'], max_det=300, verbose=False,
    )[0]
    return {
        'image': rgb,
        'ids': result.boxes.cls.cpu().numpy().astype(int).tolist(),
        'scores': result.boxes.conf.cpu().numpy().tolist(),
        'coords': result.boxes.xyxy.cpu().numpy().tolist(),
    }


def render_inference(cached, lower, upper):
    if cached is None:
        return None, 'Upload a construction-site photograph to run the committed detector.'
    lo, hi = sorted((float(lower), float(upper)))
    ids, scores, coords = cached['ids'], cached['scores'], cached['coords']
    head = max((float(s) for cid, s in zip(ids, scores) if cid == HEAD), default=0.)
    helmet = max((float(s) for cid, s in zip(ids, scores) if cid == HELMET), default=0.)
    none = max(head, helmet) < M['evidence_conf']
    if none:
        label, desc = 'REVIEW', 'Insufficient head/helmet evidence — refer to a human.'
    elif head >= hi:
        label, desc = 'FLAG', 'Potential exposed head: intervention indicated.'
    elif head < lo:
        label, desc = 'PASS', 'No exposed-head score crosses the pass boundary.'
    else:
        label, desc = 'REVIEW', 'Borderline exposed-head evidence — refer to a human.'
    colors = {'head':'#e45e48','helmet':'#17a48d','person':'#4384bd'}
    canvas = cached['image'].copy()
    draw = ImageDraw.Draw(canvas)
    for cid, score, box in zip(ids, scores, coords):
        if score < (lo if cid == HEAD else .25):
            continue
        x1,y1,x2,y2 = box
        name = NAMES[cid]
        color = colors.get(name.lower(), '#697b8c')
        draw.rectangle([x1,y1,x2,y2], outline=color, width=max(2,canvas.width//450))
        text = f'{name}  {score:.2f}'
        y = max(0,y1-17)
        draw.text((x1+2,y), text, fill=color, stroke_width=1, stroke_fill='white')
    summary = (
        f'### {label} · {desc}\n\n'
        f'**Top exposed-head score:** `{head:.3f}`   |   '
        f'**Top helmet score:** `{helmet:.3f}`   |   '
        f'**Review band:** `{lo:.3f}–{hi:.3f}`   |   '
        f'**No-evidence override:** {"ON" if none else "off"}\n\n'
        '**Interpretation:** model confidence is not a calibrated probability. '
        'An image-level pass does not certify every person as PPE-compliant.'
    )
    return canvas, summary


def table_per_class():
    return pd.DataFrame([
        {'Model': k, 'Class': v['class'], 'GT boxes': v['gt_instances'],
         'Precision @.25': v['precision_at_0.25'], 'Recall @.25': v['recall_at_0.25'],
         'AP50': v['AP50'], 'mAP50–95': v['mAP50_95']}
        for k, b in M['benchmarks'].items() for v in b['per_class']
    ])


def table_summary():
    return pd.DataFrame([
        {'Model': key, 'AP50 (custom common protocol)': val['AP50'], 'mAP50–95': val['mAP50_95']}
        for key, val in M['benchmarks'].items()
    ])


def object_cm():
    cm = np.asarray(M['benchmarks']['YOLO11s fine-tuned']['object_confusion'])
    fig, ax = plt.subplots(figsize=(7.5,4.7), constrained_layout=True)
    ax.imshow(cm, cmap='Blues')
    labels=NAMES+['Background']
    ax.set_xticks(range(4),labels)
    ax.set_yticks(range(4),labels)
    for (r,c), v in np.ndenumerate(cm):
        ax.text(c,r,f'{v:,}',ha='center',va='center',fontsize=10)
    ax.set(xlabel='Predicted',ylabel='Actual',title='Held-out object-level confusion · IoU 0.50, conf .25')
    return fig


## 4 · Gradio interface and launch

In [ ]:
# Gradio 5 styles on Blocks; Gradio 6 moved these kwargs to launch().
MODERN_GRADIO = 'css' in inspect.signature(gr.Blocks.launch).parameters
STYLE_ARGS = {} if MODERN_GRADIO else {'theme': gr.themes.Soft(), 'css': CSS}
with gr.Blocks(title='SiteSafe | PPE Decision Intelligence', **STYLE_ARGS) as demo:
    gr.HTML('''<div id="hero"><div class="eyebrow">Construction safety · model decision laboratory</div>
      <h1>SiteSafe <span style="font-weight:400;color:#b9dedb">/ PPE Monitor</span></h1>
      <p>Cost-aware detection, supervisor triage, and an auditable operating policy.</p>
      <span class="tag">YOLO11s · fine-tuned</span><span class="tag">3 actions · pass / review / flag</span>
      <span class="tag">Held-out evaluation</span><span class="tag">No training at startup</span></div>''')

    with gr.Tabs():
        with gr.Tab('01 · Operating policy'):
            gr.Markdown('### Tune the decision policy')
            gr.Markdown('Thresholds were selected on **validation**. This interactive view evaluates the fixed model on the **held-out test**; changing sliders does **not** retrain or reselect the submitted policy.')
            with gr.Row():
                lower = gr.Slider(0, 1, value=M['low_threshold'], step=.005,
                                  label='PASS boundary · lower confidence', info='Below this, pass unless evidence is missing')
                upper = gr.Slider(0, 1, value=M['high_threshold'], step=.005,
                                  label='FLAG boundary · higher confidence', info='At or above this, flag for intervention')
                prevalence = gr.Slider(1, 60, value=100*M['base_rate_assumed'], step=1,
                                        label='Assumed exposed-head prevalence (%)', info='Planning scenario; not the observed test frequency')
            kpis = gr.HTML()
            explanation = gr.Markdown()
            with gr.Row():
                cm_plot = gr.Plot(label='Policy confusion matrices')
                sweep_plot = gr.Plot(label='Validation cost curve')
            with gr.Row():
                mix_plot = gr.Plot(label='Expected actions', scale=1)
                comparison = gr.Dataframe(label='Costed alternatives', interactive=False, scale=2)
            gr.Markdown('**Important:** The €250 missed case is an illustrative expected loss. The €0.75 human check assumes perfect error correction. The cheapest unconstrained strategy may exceed the allocated 120 reviewer minutes per day.')

        with gr.Tab('02 · Inspect a photograph'):
            gr.Markdown('### Run the submitted model on a new image')
            gr.Markdown('Inference runs **once per uploaded photo**. Moving a policy slider redraws the existing detections without re-running the neural network.')
            with gr.Row():
                uploaded = gr.Image(type='pil', label='Source photo', height=430)
                annotated = gr.Image(type='pil', label='Predicted heads, helmets and people', height=430, interactive=False)
            inference_text = gr.Markdown('Upload a construction-site photograph to run the committed detector.')
            gr.Markdown('**Box colors:** orange-red = exposed head, teal = helmet, blue = person. Boxes are localization cues, not proof that every worker is safe.')

        with gr.Tab('03 · Model evaluation'):
            gr.Markdown('### Performance on the same 706-image test split')
            gr.Markdown('The positive business condition is **at least one annotated exposed head in the image**. Detection metrics below are **object-level**, whereas the decision policy uses image-level truth.')
            gr.Dataframe(value=table_summary(), interactive=False, label='Aggregate baseline comparison')
            gr.Dataframe(value=table_per_class(), interactive=False, label='Per-class precision, recall, AP, and annotation support')
            gr.Plot(value=object_cm(), label='Fine-tuned model object confusion')
            gr.Markdown(f"**Official Ultralytics test metrics:** AP50 = {M['native_ultralytics_test']['mAP50']:.4f}, mAP50–95 = {M['native_ultralytics_test']['mAP50_95']:.4f}. Custom AP differs slightly because its matching/interpolation method is not identical. The COCO zero-shot mapping includes `person` only; it does not recognize equivalent `head` or `helmet` categories.")
            gr.Markdown('### Dataset health · original Roboflow v10 split')
            gr.Dataframe(value=HEALTH, interactive=False)
            gr.Markdown('Training 4,916 images · validation 1,413 · test 706. An exact SHA-256 check found no identical files crossing splits, but cannot exclude similar video frames or sites. The `person` category has only 64 test boxes and performs poorly; detection of exposed heads remains the business target.')

    gr.HTML('<div id="footnote">SITE SAFETY SCREENING · Educational decision-support simulation, not an autonomous compliance system. All costs are illustrative and site-specific validation is required before deployment.</div>')
    inputs=[lower,upper,prevalence]
    outputs=[kpis,explanation,cm_plot,sweep_plot,mix_plot,comparison]
    demo.load(refresh,inputs=inputs,outputs=outputs)
    for ctrl in inputs:
        ctrl.change(refresh,inputs=inputs,outputs=outputs)

    cache=gr.State(value=None)
    uploaded.change(inference_once,inputs=[uploaded],outputs=[cache]).then(
        render_inference,inputs=[cache,lower,upper],outputs=[annotated,inference_text])
    for ctrl in (lower,upper):
        ctrl.change(render_inference,inputs=[cache,lower,upper],outputs=[annotated,inference_text])

if __name__=='__main__':
    demo.launch(server_name=os.getenv('APP_HOST','127.0.0.1'),
                server_port=int(os.getenv('APP_PORT','7860')),
                share=False, show_error=True,
                **({'theme': gr.themes.Soft(), 'css': CSS} if MODERN_GRADIO else {}))
